# DCGAN on CIFAR-10
**Prof. Ing. Carlos Andrés Sierra, M.Sc. — Universidad Distrital FJC**

Practical example from the *Deep Learning: Advanced Topics* slides — *Generative Adversarial Networks* section.

Three controlled experiments on **CIFAR-10** (60 000 RGB images, 10 classes, 32×32):

| Experiment | Architecture | Loss | Goal |
|------------|-------------|------|------|
| **Exp A** | DCGAN | BCE | Baseline; record FID at regular intervals |
| **Exp B** | DCGAN | WGAN-GP | Compare training stability vs. Exp A |
| **Exp C** | cGAN (conditional) | WGAN-GP | Class-conditional generation |

> **Expected result:** WGAN-GP achieves lower FID than standard DCGAN; cGAN generates recognisable class-conditional samples.  
> **Architecture:** Generator: z ∈ ℝ¹²⁸ → reshape 4×4 → 4 ConvTranspose blocks → 32×32×3.  Discriminator / Critic: 32×32×3 → 4 strided Conv blocks → scalar.

---
## Section 1 — Install & Import Libraries

In [ ]:
import sys, subprocess

_pkgs = [
    'torch', 'torchvision', 'numpy<2',
    'matplotlib', 'scipy', 'tqdm',
]
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--prefer-binary'] + _pkgs
)
print('Packages ready.')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models import inception_v3
from scipy import linalg
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
Z_DIM       = 128          # noise vector dimension
N_CLASSES   = 10           # CIFAR-10 classes
BATCH_SIZE  = 128
EPOCHS      = 30           # increase to 100+ for publication-quality results
FID_EVERY   = 10           # compute FID every N epochs
N_FID       = 2000         # real/fake samples used for FID estimation
GP_LAMBDA   = 10.0         # gradient penalty coefficient (Exp B, C)
CRITIC_ITER = 5            # critic updates per generator update (WGAN-GP)
LR_G        = 2e-4
LR_D        = 2e-4
BETAS       = (0.5, 0.999)

os.makedirs('./outputs', exist_ok=True)
print(f'Using device: {DEVICE}')
print(f'Z_DIM={Z_DIM}  BATCH_SIZE={BATCH_SIZE}  EPOCHS={EPOCHS}')

---
## Section 2 — Load CIFAR-10

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # → [-1, 1]
])

train_set = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=(DEVICE == 'cuda'), drop_last=True
)

CLASS_NAMES = train_set.classes
print(f'Training samples : {len(train_set):,}')
print(f'Classes          : {CLASS_NAMES}')

In [ ]:
# Show one sample per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
shown = {c: False for c in range(N_CLASSES)}
for img, label in train_set:
    label = int(label)
    if not shown[label]:
        ax = axes[label // 5][label % 5]
        ax.imshow(img.permute(1, 2, 0).numpy() * 0.5 + 0.5)
        ax.set_title(CLASS_NAMES[label], fontsize=9)
        ax.axis('off')
        shown[label] = True
    if all(shown.values()):
        break
fig.suptitle('CIFAR-10 — one sample per class', fontsize=12)
plt.tight_layout()
plt.savefig('./outputs/cifar10_samples.png', dpi=120)
plt.show()
print('Saved: outputs/cifar10_samples.png')

---
## Section 3 — Architecture Definitions

### 3a · Shared Blocks

In [ ]:
def weights_init(m):
    """DCGAN weight initialisation (mean=0, std=0.02)."""
    cname = m.__class__.__name__
    if 'Conv' in cname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)


def conv_block(in_ch, out_ch, stride=2, bn=True):
    """Discriminator strided-conv block."""
    layers = [nn.Conv2d(in_ch, out_ch, 4, stride, 1, bias=not bn)]
    if bn:
        layers.append(nn.BatchNorm2d(out_ch))
    layers.append(nn.LeakyReLU(0.2, inplace=True))
    return nn.Sequential(*layers)


def deconv_block(in_ch, out_ch, bn=True):
    """Generator transposed-conv block."""
    layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=not bn)]
    if bn:
        layers.append(nn.BatchNorm2d(out_ch))
    layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

### 3b · Generator & Discriminator (DCGAN / WGAN-GP)

In [ ]:
class Generator(nn.Module):
    """
    z ∈ ℝ^Z_DIM  →  FC → reshape 512×2×2
    → 4 ConvTranspose blocks (512→256→128→64→3)
    Block sizes: 2→4→8→16→32 ; Output: 3×32×32, tanh activation
    """
    def __init__(self, z_dim=Z_DIM, n_class=0):
        super().__init__()
        self.use_cond = n_class > 0
        in_dim = z_dim + n_class if self.use_cond else z_dim
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 512 * 2 * 2), nn.ReLU(inplace=True)
        )
        self.conv = nn.Sequential(
            deconv_block(512, 256),   # 2×2  → 4×4
            deconv_block(256, 128),   # 4×4  → 8×8
            deconv_block(128, 64),    # 8×8  → 16×16
            # Block 4: output — no BN, tanh
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=True),  # 16×16 → 32×32
            nn.Tanh(),
        )

    def forward(self, z, labels=None):
        if self.use_cond and labels is not None:
            one_hot = torch.zeros(z.size(0), N_CLASSES, device=z.device)
            one_hot.scatter_(1, labels.unsqueeze(1), 1)
            z = torch.cat([z, one_hot], dim=1)
        x = self.fc(z).view(-1, 512, 2, 2)
        return self.conv(x)


class Discriminator(nn.Module):
    """
    3×32×32  →  4 strided Conv blocks (64→128→256→512)  →  scalar
    Block sizes: 32→16→8→4→2 ; fc input: 512×2×2 = 2048
    sigmoid=False → WGAN-GP Critic (unbounded output)
    """
    def __init__(self, sigmoid=True, n_class=0):
        super().__init__()
        self.use_cond = n_class > 0
        in_ch = 3 + n_class if self.use_cond else 3
        self.conv = nn.Sequential(
            conv_block(in_ch, 64,  bn=False),   # 32×32 → 16×16 (no BN on first layer)
            conv_block(64,  128),               # 16×16 → 8×8
            conv_block(128, 256),               # 8×8   → 4×4
            conv_block(256, 512),               # 4×4   → 2×2
        )
        self.fc = nn.Linear(512 * 2 * 2, 1)
        self.out_act = nn.Sigmoid() if sigmoid else nn.Identity()

    def forward(self, x, labels=None):
        if self.use_cond and labels is not None:
            # Broadcast one-hot label map across spatial dims
            lbl_map = torch.zeros(x.size(0), N_CLASSES, *x.shape[2:], device=x.device)
            lbl_map.scatter_(1, labels.view(-1, 1, 1, 1).expand(-1, N_CLASSES, *x.shape[2:]), 1)
            x = torch.cat([x, lbl_map], dim=1)
        feat = self.conv(x)
        return self.out_act(self.fc(feat.view(feat.size(0), -1)))


# Sanity check — verify output shapes
G_test = Generator()
D_test = Discriminator()
z_test = torch.randn(4, Z_DIM)
fake_test = G_test(z_test)
score_test = D_test(fake_test)
assert fake_test.shape == (4, 3, 32, 32), f"G output shape: {fake_test.shape}"
assert score_test.shape == (4, 1),        f"D output shape: {score_test.shape}"
G_params = sum(p.numel() for p in G_test.parameters())
D_params = sum(p.numel() for p in D_test.parameters())
print(f'Generator      : {G_params:,} parameters  | output shape: {tuple(fake_test.shape)}')
print(f'Discriminator  : {D_params:,} parameters  | output shape: {tuple(score_test.shape)}')
del G_test, D_test, z_test, fake_test, score_test

---
## Section 4 — FID Computation Helper

FID compares Inception-v3 pool-layer feature statistics between real and generated images.  
**Lower FID = better image quality and diversity.**

In [ ]:
@torch.no_grad()
def get_inception_features(imgs_tensor, model, batch=64):
    """Pass imgs (N,3,H,W) in [-1,1] through InceptionV3 pool layer → (N,2048)."""
    feats = []
    imgs_up = torch.nn.functional.interpolate(
        imgs_tensor, size=(299, 299), mode='bilinear', align_corners=False
    )
    imgs_up = (imgs_up + 1) / 2  # rescale → [0, 1] for Inception
    for i in range(0, len(imgs_up), batch):
        out = model(imgs_up[i:i+batch].to(DEVICE))
        feats.append(out.squeeze(-1).squeeze(-1).cpu().numpy())
    return np.concatenate(feats, axis=0)


def compute_fid(real_feats, fake_feats):
    mu_r, sig_r = real_feats.mean(0), np.cov(real_feats, rowvar=False)
    mu_g, sig_g = fake_feats.mean(0), np.cov(fake_feats, rowvar=False)
    diff = mu_r - mu_g
    cov_sqrt, _ = linalg.sqrtm(sig_r @ sig_g, disp=False)
    if np.iscomplexobj(cov_sqrt):
        cov_sqrt = cov_sqrt.real
    return float(diff @ diff + np.trace(sig_r + sig_g - 2 * cov_sqrt))


# Load InceptionV3, strip classifier → 2048-D pool features
inception = inception_v3(weights='DEFAULT')
inception.fc = nn.Identity()
inception.aux_logits = False
inception = inception.to(DEVICE).eval()

# Pre-compute real features once (reused across all experiments)
print(f'Pre-computing real Inception features on {N_FID} samples ...')
real_imgs_list = []
for imgs, _ in train_loader:
    real_imgs_list.append(imgs)
    if sum(x.size(0) for x in real_imgs_list) >= N_FID:
        break
real_imgs_all = torch.cat(real_imgs_list, dim=0)[:N_FID]
REAL_FEATS = get_inception_features(real_imgs_all, inception)
print(f'Real feature matrix: {REAL_FEATS.shape}')

---
## Section 5 — Training Helpers

In [ ]:
def sample_noise(n, z_dim=Z_DIM):
    return torch.randn(n, z_dim, device=DEVICE)


def gradient_penalty(critic, real, fake, labels=None):
    """WGAN-GP gradient penalty (Gulrajani et al., 2017)."""
    alpha = torch.rand(real.size(0), 1, 1, 1, device=DEVICE)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = critic(interp, labels)
    grad = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True
    )[0]
    return ((grad.norm(2, dim=[1, 2, 3]) - 1) ** 2).mean()


@torch.no_grad()
def generate_grid(G, n=64, labels=None):
    """Return an 8×8 grid tensor of generated images."""
    z = sample_noise(n)
    imgs = G(z, labels[:n].to(DEVICE) if labels is not None else None)
    imgs = (imgs * 0.5 + 0.5).clamp(0, 1)
    return torchvision.utils.make_grid(imgs, nrow=8, padding=2)


@torch.no_grad()
def fid_for_model(G, n=N_FID, conditional=False):
    fake_list = []
    while sum(x.size(0) for x in fake_list) < n:
        z = sample_noise(BATCH_SIZE)
        lbl = torch.randint(0, N_CLASSES, (BATCH_SIZE,), device=DEVICE) if conditional else None
        fake_list.append(G(z, lbl).cpu())
    fake_imgs = torch.cat(fake_list, dim=0)[:n]
    fake_feats = get_inception_features(fake_imgs, inception)
    return compute_fid(REAL_FEATS, fake_feats)


# ── Exp A: DCGAN + BCE ────────────────────────────────────────────────────────
def train_dcgan(epochs=EPOCHS, label='expA'):
    G = Generator().to(DEVICE).apply(weights_init)
    D = Discriminator(sigmoid=True).to(DEVICE).apply(weights_init)
    opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=BETAS)
    opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=BETAS)
    criterion = nn.BCELoss()
    history = {'d_loss': [], 'g_loss': [], 'fid': [], 'fid_epoch': []}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        d_buf, g_buf = [], []
        for real, _ in tqdm(train_loader, desc=f'[{label}] E{epoch:02d}', leave=False):
            real = real.to(DEVICE)
            bs   = real.size(0)
            real_lbl = torch.full((bs, 1), 0.9, device=DEVICE)   # label smoothing
            fake_lbl = torch.zeros(bs, 1, device=DEVICE)

            opt_D.zero_grad()
            loss_D = criterion(D(real), real_lbl) + criterion(D(G(sample_noise(bs)).detach()), fake_lbl)
            loss_D.backward(); opt_D.step()

            opt_G.zero_grad()
            loss_G = criterion(D(G(sample_noise(bs))), real_lbl)
            loss_G.backward(); opt_G.step()

            d_buf.append(loss_D.item()); g_buf.append(loss_G.item())

        history['d_loss'].append(np.mean(d_buf))
        history['g_loss'].append(np.mean(g_buf))

        if epoch % FID_EVERY == 0 or epoch == epochs:
            fid = fid_for_model(G)
            history['fid'].append(fid); history['fid_epoch'].append(epoch)
            print(f'[{label}] Epoch {epoch:3d} | D={history["d_loss"][-1]:.4f} '
                  f'| G={history["g_loss"][-1]:.4f} | FID={fid:.1f}')

    return G, D, history


# ── Exp B & C: WGAN-GP (unconditional / conditional) ─────────────────────────
def train_wgan_gp(epochs=EPOCHS, conditional=False, label='expB'):
    n_cls = N_CLASSES if conditional else 0
    G = Generator(n_class=n_cls).to(DEVICE).apply(weights_init)
    D = Discriminator(sigmoid=False, n_class=n_cls).to(DEVICE).apply(weights_init)
    opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=BETAS)
    opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=BETAS)
    history = {'d_loss': [], 'g_loss': [], 'fid': [], 'fid_epoch': []}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        d_buf, g_buf = [], []

        for real, lbls in tqdm(train_loader, desc=f'[{label}] E{epoch:02d}', leave=False):
            real = real.to(DEVICE)
            lbls = lbls.to(DEVICE) if conditional else None
            bs   = real.size(0)

            for _ in range(CRITIC_ITER):
                fake_lbl = torch.randint(0, N_CLASSES, (bs,), device=DEVICE) if conditional else None
                with torch.no_grad():
                    fake = G(sample_noise(bs), fake_lbl)
                opt_D.zero_grad()
                loss_D = (-D(real, lbls).mean() + D(fake, fake_lbl).mean()
                          + GP_LAMBDA * gradient_penalty(D, real, fake.detach(), lbls))
                loss_D.backward(); opt_D.step()
                d_buf.append(loss_D.item())

            fake_lbl = torch.randint(0, N_CLASSES, (bs,), device=DEVICE) if conditional else None
            opt_G.zero_grad()
            loss_G = -D(G(sample_noise(bs), fake_lbl), fake_lbl).mean()
            loss_G.backward(); opt_G.step()
            g_buf.append(loss_G.item())

        history['d_loss'].append(np.mean(d_buf))
        history['g_loss'].append(np.mean(g_buf))

        if epoch % FID_EVERY == 0 or epoch == epochs:
            fid = fid_for_model(G, conditional=conditional)
            history['fid'].append(fid); history['fid_epoch'].append(epoch)
            print(f'[{label}] Epoch {epoch:3d} | D={history["d_loss"][-1]:.4f} '
                  f'| G={history["g_loss"][-1]:.4f} | FID={fid:.1f}')

    return G, D, history


print('Training functions defined.')

---
## Section 6 — Experiment A: DCGAN with BCE Loss

In [ ]:
# Re-register Generator and Discriminator with the corrected architecture.
# Root cause: the original Generator started at 4×4 and produced 64×64 images
# (4 upsamplings: 4→8→16→32→64), but CIFAR-10 is 32×32.
# Fix: start at 2×2 so 4 ConvTranspose blocks land exactly at 32×32 (2→4→8→16→32).
# Discriminator fc stays at 512*2*2=2048 because 32×32 input gives 2×2 after 4 stride-2 blocks.

class Generator(nn.Module):
    """z ∈ ℝ^128 → FC → 512×2×2 → 4 ConvTranspose blocks → 3×32×32"""
    def __init__(self, z_dim=Z_DIM, n_class=0):
        super().__init__()
        self.use_cond = n_class > 0
        in_dim = z_dim + n_class if self.use_cond else z_dim
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 512 * 2 * 2), nn.ReLU(inplace=True)
        )
        self.conv = nn.Sequential(
            deconv_block(512, 256),                              # 2×2  → 4×4
            deconv_block(256, 128),                              # 4×4  → 8×8
            deconv_block(128, 64),                               # 8×8  → 16×16
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=True),      # 16×16 → 32×32
            nn.Tanh(),
        )

    def forward(self, z, labels=None):
        if self.use_cond and labels is not None:
            one_hot = torch.zeros(z.size(0), N_CLASSES, device=z.device)
            one_hot.scatter_(1, labels.unsqueeze(1), 1)
            z = torch.cat([z, one_hot], dim=1)
        return self.conv(self.fc(z).view(-1, 512, 2, 2))


class Discriminator(nn.Module):
    """3×32×32 → 4 strided Conv blocks → 512×2×2 → fc → scalar"""
    def __init__(self, sigmoid=True, n_class=0):
        super().__init__()
        self.use_cond = n_class > 0
        in_ch = 3 + n_class if self.use_cond else 3
        self.conv = nn.Sequential(
            conv_block(in_ch, 64,  bn=False),   # 32×32 → 16×16
            conv_block(64,  128),               # 16×16 → 8×8
            conv_block(128, 256),               # 8×8   → 4×4
            conv_block(256, 512),               # 4×4   → 2×2
        )
        self.fc = nn.Linear(512 * 2 * 2, 1)
        self.out_act = nn.Sigmoid() if sigmoid else nn.Identity()

    def forward(self, x, labels=None):
        if self.use_cond and labels is not None:
            lbl_map = torch.zeros(x.size(0), N_CLASSES, *x.shape[2:], device=x.device)
            lbl_map.scatter_(1, labels.view(-1, 1, 1, 1).expand(-1, N_CLASSES, *x.shape[2:]), 1)
            x = torch.cat([x, lbl_map], dim=1)
        feat = self.conv(x)
        return self.out_act(self.fc(feat.view(feat.size(0), -1)))


# Verify shapes before training
_z = torch.randn(2, Z_DIM)
_fake = Generator()(_z)
_score = Discriminator()(_fake)
assert _fake.shape  == (2, 3, 32, 32), f"G output: {_fake.shape}"
assert _score.shape == (2, 1),         f"D output: {_score.shape}"
print(f"Generator  output : {tuple(_fake.shape)}  ✓")
print(f"Discriminator output : {tuple(_score.shape)}  ✓")
del _z, _fake, _score

In [ ]:
print('=' * 60)
print('EXPERIMENT A — DCGAN + BCE (baseline)')
print('=' * 60)
G_A, D_A, hist_A = train_dcgan(epochs=EPOCHS, label='expA')

In [ ]:
# Loss curves — Exp A
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(hist_A['d_loss'], label='D loss', color='steelblue')
axes[0].plot(hist_A['g_loss'], label='G loss', color='tomato')
axes[0].set_title('Exp A — Training Losses (BCE)')
axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(hist_A['fid_epoch'], hist_A['fid'], marker='o', color='purple')
axes[1].set_title('Exp A — FID over Training')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FID ↓')
plt.tight_layout()
plt.savefig('./outputs/expA_curves.png', dpi=120)
plt.show()
print('Saved: outputs/expA_curves.png')

In [ ]:
# Generated image grid — Exp A
G_A.eval()
grid_A = generate_grid(G_A)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(grid_A.permute(1, 2, 0).cpu().numpy())
ax.axis('off')
ax.set_title(f'Exp A — DCGAN BCE — Epoch {EPOCHS}  (FID={hist_A["fid"][-1]:.1f})')
plt.tight_layout()
plt.savefig('./outputs/expA_grid.png', dpi=120)
plt.show()
print('Saved: outputs/expA_grid.png')

---
## Section 7 — Experiment B: WGAN-GP (unconditional)
Same architecture; BCE loss replaced by Wasserstein-1 distance + gradient penalty.  
The critic is updated **5×** per generator step to satisfy the Lipschitz constraint.

In [ ]:
print('=' * 60)
print('EXPERIMENT B — WGAN-GP (stability comparison)')
print('=' * 60)
G_B, D_B, hist_B = train_wgan_gp(epochs=EPOCHS, conditional=False, label='expB')

In [ ]:
# Loss curves — Exp B
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(hist_B['d_loss'], label='Critic loss', color='steelblue')
axes[0].plot(hist_B['g_loss'], label='G loss', color='tomato')
axes[0].set_title('Exp B — Training Losses (WGAN-GP)')
axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(hist_B['fid_epoch'], hist_B['fid'], marker='o', color='purple')
axes[1].set_title('Exp B — FID over Training')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FID ↓')
plt.tight_layout()
plt.savefig('./outputs/expB_curves.png', dpi=120)
plt.show()
print('Saved: outputs/expB_curves.png')

In [ ]:
# Generated image grid — Exp B
G_B.eval()
grid_B = generate_grid(G_B)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(grid_B.permute(1, 2, 0).cpu().numpy())
ax.axis('off')
ax.set_title(f'Exp B — WGAN-GP — Epoch {EPOCHS}  (FID={hist_B["fid"][-1]:.1f})')
plt.tight_layout()
plt.savefig('./outputs/expB_grid.png', dpi=120)
plt.show()
print('Saved: outputs/expB_grid.png')

---
## Section 8 — Experiment C: Conditional GAN (cGAN + WGAN-GP)
Class label **y** is concatenated with **z** in G (one-hot, 10-D) and projected as a spatial label map in D.

In [ ]:
print('=' * 60)
print('EXPERIMENT C — cGAN + WGAN-GP (class-conditional)')
print('=' * 60)
G_C, D_C, hist_C = train_wgan_gp(epochs=EPOCHS, conditional=True, label='expC')

In [ ]:
# Class-conditional grid: 10 rows × 8 columns, one row per class
G_C.eval()
z_c   = sample_noise(80)
lbl_c = torch.arange(N_CLASSES, device=DEVICE).repeat_interleave(8)
with torch.no_grad():
    fake_c = G_C(z_c, lbl_c)
fake_c = (fake_c * 0.5 + 0.5).clamp(0, 1)
grid_C = torchvision.utils.make_grid(fake_c, nrow=8, padding=2)

fig, ax = plt.subplots(figsize=(10, 13))
ax.imshow(grid_C.permute(1, 2, 0).cpu().numpy())
ax.axis('off')
row_h = grid_C.shape[1] / N_CLASSES
for i, name in enumerate(CLASS_NAMES):
    ax.text(-4, i * row_h + row_h / 2, name, va='center', ha='right',
            fontsize=8, color='navy', fontweight='bold')
ax.set_title(f'Exp C — cGAN — Epoch {EPOCHS}  (FID={hist_C["fid"][-1]:.1f})\n'
             '8 samples per class (rows = classes)')
plt.tight_layout()
plt.savefig('./outputs/expC_class_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/expC_class_grid.png')

In [ ]:
# Loss curves — Exp C
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(hist_C['d_loss'], label='Critic loss', color='steelblue')
axes[0].plot(hist_C['g_loss'], label='G loss', color='tomato')
axes[0].set_title('Exp C — Training Losses (cGAN + WGAN-GP)')
axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(hist_C['fid_epoch'], hist_C['fid'], marker='o', color='purple')
axes[1].set_title('Exp C — FID over Training')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FID ↓')
plt.tight_layout()
plt.savefig('./outputs/expC_curves.png', dpi=120)
plt.show()
print('Saved: outputs/expC_curves.png')

---
## Section 9 — Cross-Experiment Comparison

In [ ]:
# FID comparison and side-by-side grids
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for hist, lbl, col in [
    (hist_A, 'Exp A — DCGAN + BCE',     'steelblue'),
    (hist_B, 'Exp B — WGAN-GP',          'tomato'),
    (hist_C, 'Exp C — cGAN + WGAN-GP',  'seagreen'),
]:
    axes[0].plot(hist['fid_epoch'], hist['fid'], marker='o', label=lbl, color=col)
axes[0].set_title('FID Comparison Across Experiments')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('FID ↓  (lower is better)')
axes[0].legend()

# Stack final grids horizontally
combined = torchvision.utils.make_grid(
    [grid_A.unsqueeze(0), grid_B.unsqueeze(0), grid_C.unsqueeze(0)],
    nrow=3, padding=4
).squeeze()
axes[1].imshow(combined.permute(1, 2, 0).cpu().numpy())
axes[1].axis('off')
axes[1].set_title(
    f'A (BCE) FID={hist_A["fid"][-1]:.0f}   '
    f'B (WGAN-GP) FID={hist_B["fid"][-1]:.0f}   '
    f'C (cGAN) FID={hist_C["fid"][-1]:.0f}',
    fontsize=9
)

plt.tight_layout()
plt.savefig('./outputs/comparison.png', dpi=120)
plt.show()
print('Saved: outputs/comparison.png')

---
## Summary

| Experiment | Loss | Key observation |
|------------|------|-----------------|
| **A — DCGAN + BCE** | Binary Cross-Entropy | Baseline; D/G losses can oscillate; FID degrades if mode collapse occurs |
| **B — WGAN-GP** | Wasserstein-1 + GP (λ=10) | Smoother critic loss; more stable training; lower FID than Exp A |
| **C — cGAN + WGAN-GP** | Wasserstein-1 + GP (λ=10) | Class-conditional rows show recognisable object texture and colour |

**Key take-aways:**
1. **Wasserstein distance** always provides meaningful gradients — the critic loss is a reliable proxy for image quality, unlike BCE which saturates when D wins.
2. **Gradient penalty** (λ=10) enforces the Lipschitz constraint more stably than weight clipping; training becomes predictable enough to monitor with FID.
3. **Conditional generation** requires only a minor architectural change: concatenate the one-hot label with z in G, and broadcast a label map spatially in D.
4. For **30 epochs**, all three experiments will produce blurry but class-distinguishable images. Run 100+ epochs (with a GPU) to see the FID gap between A and B/C widen significantly.

**Connection to Challenge 4 (GAIL):** the DCGAN discriminator trained here is conceptually identical to the GAIL discriminator — it separates expert trajectories from policy trajectories, replacing the environment reward signal.